In [ ]:
import os

from glob import glob
from PIL import Image
from rich.pretty import pprint
from pydantic_ai import BinaryContent

from common.cache import RedisCache
from istm.llm_agents import ImageDescriber, ImageDescriberInput

In [ ]:
def show_image(image_path: str) -> None:
    return Image.open(image_path)

In [ ]:
def get_binary_content(image_path: str) -> BinaryContent:
    with open(image_path, "rb") as image_file:
        return BinaryContent(
            data=image_file.read(),
            media_type="image/jpg",
        )

In [ ]:
image_paths = glob("/resources/resized-images/*.jpg")
print(f"image_paths => {len(image_paths)}")

In [ ]:
description_guidelines = (
    "Carefully analyze the provided image and generate a vivid, highly detailed description. "
    "Use rich, expressive language to convey the style, composition, atmosphere, and visual elements. "
    "Always begin the description with 'a...' — note that it should start in lowercase. "
    "Although the image is an illustration, describe it as if it were a photograph, and do not mention the word 'illustration'."
)

agent_inputs = [
    ImageDescriberInput(description_guidelines=description_guidelines)
    for _ in image_paths
]

user_contents = [
    get_binary_content(image_path=image_path) for image_path in image_paths
]

image_describer = ImageDescriber(
    cache=RedisCache(),
    max_concurrency=5,
)

image_describer_outputs = await image_describer.batch_generate(
    agent_inputs=agent_inputs,
    user_contents=user_contents,
)

In [ ]:
idx = 10

print(image_describer_outputs[idx].description)
Image.open(image_paths[idx])

In [ ]:
out_base_path = "/resources/cropped-train-images"
for image_path, image_describer_output in zip(
    image_paths,
    image_describer_outputs,
):
    out_file = os.path.basename(image_path).replace(".jpg", ".txt")
    out_path = f"{out_base_path}/{out_file}"
    with open(out_path, "w") as file:
        description = (
            f"In the style of GRCRA, {image_describer_output.description}"
        )
        file.write(description)
